# MLB S26 Hackathon — Group 2 (Self-Contained Colab Notebook)

Zero-shot protein fitness prediction using **ESM-2 LLR + ESM-IF + Ridge calibration on Round-1 queried labels**.

**Best public leaderboard Spearman ρ = 0.41266**

This notebook is fully self-contained: just open it in Google Colab and **Runtime → Run all**.
It will:
1. Install dependencies
2. Clone the project repo (which already contains all cached features and data)
3. Reproduce the best leaderboard submission (`calibrate_q1.py` logic, inlined)
4. Optionally re-compute the ESM-2 features from scratch (Section 4 — requires GPU)
5. Optionally run the broader `predict_v3.py` ablation (Section 5)
6. Optionally run the MSA-Transformer blend (Section 6)

> **TA note:** No manual file uploads are required. All `.npy` cache features and the `Hackathon_data/` CSVs are committed to the repo, so Sections 1–3 reproduce the best result without any GPU.

## 1. Install dependencies

We need `fair-esm` (Facebook ESM models), plus the standard scientific stack. `torch`, `numpy`, `pandas`, `scikit-learn`, `scipy` are already on Colab; we install/upgrade the rest.

In [ ]:
!pip install -q fair-esm
# torch, numpy, pandas, scikit-learn, scipy ship with Colab — install only if missing
!pip install -q --upgrade numpy pandas scikit-learn scipy

## 2. Clone the project repo

The repo includes `Hackathon_data/` (train/test/queried_round1 CSVs + WT FASTA) and `esm_cache/` (all pre-computed `.npy` log-probability tensors and pLDDT array). Cloning gives a TA everything they need to reproduce the best score in seconds, no GPU required.

In [ ]:
import os
REPO_URL = "https://github.com/ivy3h/mlb-hackathon-s26.git"
REPO_DIR = "mlb-hackathon-s26"
if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL}
os.chdir(REPO_DIR)
print("Working dir:", os.getcwd())
!ls Hackathon_data esm_cache

## 3. Reproduce the best submission (ρ ≈ 0.41266)

This section is the inlined version of `calibrate_q1.py`. It does the following:

1. **Per-mutation LLR** for each ESM-2 model (8M, 35M, 150M, 650M) using the cached masked-marginal log-probs.
2. **Median across the 4 ESM-2 models**, blended 85/15 with the **ESM-IF** structure-conditioned LLR.
3. **Entropy weighting**: down-weight evolutionarily variable positions (`1 / (1 + 0.3 · entropy[pos])`).
4. **Ridge calibration** on Round-1 queried labels (100 mutations) using `[base, low_pLDDT, model_disagreement]`.
5. **Blend** calibrated and uncalibrated predictions at `w ∈ {0.1, 0.2, 0.3}` to reduce overfitting to 100 labels.

The `w=0.20` blend is the submission that achieved the best public-leaderboard score.

In [ ]:
import os, re, sys, types
import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from sklearn.linear_model import Ridge

# Stub out esm.inverse_folding to avoid GLIBC import issues — we only need the alphabet
sys.modules['esm.inverse_folding'] = types.ModuleType('esm.inverse_folding')
import esm

DATA_DIR = "Hackathon_data"
CACHE_DIR = "esm_cache"
MODELS = [
    "esm2_t6_8M_UR50D",
    "esm2_t12_35M_UR50D",
    "esm2_t30_150M_UR50D",
    "esm2_t33_650M_UR50D",
]

def get_aa_to_tok():
    _, alphabet = esm.pretrained.esm2_t6_8M_UR50D()
    aa_list = "ACDEFGHIKLMNPQRSTVWY"
    return {aa: alphabet.get_idx(aa) for aa in aa_list}

def parse_mutants(mutants):
    wt, pos, mut = [], [], []
    for code in mutants:
        m = re.match(r'^([A-Z])(\d+)([A-Z])$', code)
        wt.append(m.group(1))
        pos.append(int(m.group(2)))
        mut.append(m.group(3))
    return wt, np.array(pos, dtype=np.int64), mut

def llr_from_logprobs(logprobs, pos, wt_tok, mut_tok):
    return logprobs[pos, mut_tok] - logprobs[pos, wt_tok]

def compute_base_and_disp(mutants, aa_to_tok, logprobs_dict, esmif_lp, entropy):
    wt, pos, mut = parse_mutants(mutants)
    wt_tok = np.array([aa_to_tok[a] for a in wt], dtype=np.int64)
    mut_tok = np.array([aa_to_tok[a] for a in mut], dtype=np.int64)

    llrs = []
    for m in MODELS:
        lp = logprobs_dict[m]
        llrs.append(llr_from_logprobs(lp, pos, wt_tok, mut_tok))
    llrs = np.vstack(llrs).T  # (n, 4)

    if_llr = llr_from_logprobs(esmif_lp, pos, wt_tok, mut_tok)
    ew = 1.0 / (1.0 + 0.3 * entropy[pos])

    med4 = np.median(llrs, axis=1)
    base = ew * (0.85 * med4 + 0.15 * if_llr)
    disp = llrs.std(axis=1)
    return base, disp, pos

# --- Load data and cached features ---
train_df = pd.read_csv(f"{DATA_DIR}/train.csv")
test_df  = pd.read_csv(f"{DATA_DIR}/test.csv")
q1_df    = pd.read_csv(f"{DATA_DIR}/queried_round1.csv")
print(f"Train={len(train_df)}, Test={len(test_df)}, Q1={len(q1_df)}")

aa_to_tok = get_aa_to_tok()
logprobs_dict = {m: np.load(f"{CACHE_DIR}/{m}_logprobs.npy") for m in MODELS}
esmif_lp = np.load(f"{CACHE_DIR}/esmif_logprobs.npy")
entropy  = np.load(f"{CACHE_DIR}/esm2_entropy.npy")
plddt    = np.load(f"{CACHE_DIR}/esmfold_plddt.npy")

train_base, train_disp, train_pos = compute_base_and_disp(train_df['mutant'].tolist(), aa_to_tok, logprobs_dict, esmif_lp, entropy)
test_base,  test_disp,  test_pos  = compute_base_and_disp(test_df['mutant'].tolist(),  aa_to_tok, logprobs_dict, esmif_lp, entropy)
q1_base,    q1_disp,    q1_pos    = compute_base_and_disp(q1_df['mutant'].tolist(),    aa_to_tok, logprobs_dict, esmif_lp, entropy)

def build_X(base, disp, pos):
    low_plddt = 1.0 - plddt[pos] / 100.0
    return np.vstack([base, low_plddt, disp]).T

X_q1 = build_X(q1_base, q1_disp, q1_pos)
y_q1 = q1_df['DMS_score'].values

mdl = Ridge(alpha=1.0)
mdl.fit(X_q1, y_q1)

X_train = build_X(train_base, train_disp, train_pos)
X_test  = build_X(test_base,  test_disp,  test_pos)
train_cal = mdl.predict(X_train)
test_cal  = mdl.predict(X_test)

print(f"Train rho base: {spearmanr(train_df['DMS_score'].values, train_base).statistic:.4f}")
print(f"Train rho cal : {spearmanr(train_df['DMS_score'].values, train_cal).statistic:.4f}")

os.makedirs("results/kaggle", exist_ok=True)
for w in [0.1, 0.2, 0.3]:
    train_blend = (1 - w) * train_base + w * train_cal
    test_blend  = (1 - w) * test_base  + w * test_cal
    rho = spearmanr(train_df['DMS_score'].values, train_blend).statistic
    tag = f"q1cal_w{int(w*100):02d}"
    pd.DataFrame({"id": np.arange(len(test_df)), "DMS_score": test_blend}).to_csv(f"results/kaggle/kaggle_{tag}.csv", index=False)
    pd.DataFrame({"mutant": test_df['mutant'], "DMS_score_predicted": test_blend}).to_csv(f"predictions_{tag}.csv", index=False)
    print(f"w={w:.2f} -> train rho {rho:.4f}  ->  predictions_{tag}.csv")

print("\nBest public-leaderboard submission: predictions_q1cal_w20.csv (Spearman ρ ≈ 0.41266)")

### 3a. Inspect & visualize the predictions

In [ ]:
import matplotlib.pyplot as plt
best = pd.read_csv("predictions_q1cal_w20.csv")
print(best.head())
print(best.describe())

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].hist(best['DMS_score_predicted'], bins=60)
axes[0].set_title('Test predictions distribution (q1cal_w20)')
axes[0].set_xlabel('Predicted DMS score')
axes[1].scatter(train_base, train_df['DMS_score'].values, s=4, alpha=0.4)
axes[1].set_xlabel('Train base score (ESM2 med + IF)')
axes[1].set_ylabel('Train DMS score')
axes[1].set_title(f"Train fit (ρ={spearmanr(train_df['DMS_score'], train_base).statistic:.3f})")
plt.tight_layout(); plt.show()

## 4. (Optional) Re-compute ESM-2 features from scratch — **GPU required**

Sections 1–3 use cached `.npy` features that are committed to the repo. If you want to fully reproduce them from scratch, this section runs `compute_esm_scores.py` (masked-marginal log-probs for the 4 ESM-2 models) and `compute_entropy.py`.

On a Colab T4/A100 this takes a few minutes for the small models and ~10–20 min for the 650M model. **Skip this section if you only want to verify the best result.**

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# Re-import esm WITHOUT the inverse_folding stub used in section 3.
# The stub above only affects code that imports esm.inverse_folding; pretrained loaders work either way.
import time
import numpy as np
import torch
import esm

DATA_DIR = "Hackathon_data"
CACHE_DIR = "esm_cache"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def load_wt_sequence(fasta_path):
    lines = []
    with open(fasta_path) as f:
        for line in f:
            if not line.startswith(">"):
                lines.append(line.strip())
    return "".join(lines)

def compute_masked_marginals(wt_seq, model_name, overwrite=False):
    cache_file = os.path.join(CACHE_DIR, f"{model_name}_logprobs.npy")
    if os.path.exists(cache_file) and not overwrite:
        print(f"[{model_name}] cached -> {cache_file}")
        return np.load(cache_file)

    print(f"Loading {model_name} ...")
    model, alphabet = getattr(esm.pretrained, model_name)()
    batch_converter = alphabet.get_batch_converter()
    model.eval().to(device)
    mask_idx = alphabet.mask_idx
    seq_len = len(wt_seq)
    vocab_size = len(alphabet)
    logprobs_matrix = np.zeros((seq_len, vocab_size), dtype=np.float32)
    batch_size = 32 if torch.cuda.is_available() else 8

    t0 = time.time()
    for start in range(0, seq_len, batch_size):
        end = min(start + batch_size, seq_len)
        batch_data = [(f"p{p}", wt_seq) for p in range(start, end)]
        _, _, tokens = batch_converter(batch_data)
        for i, p in enumerate(range(start, end)):
            tokens[i, p + 1] = mask_idx
        with torch.no_grad():
            result = model(tokens.to(device))
        log_probs = torch.log_softmax(result["logits"], dim=-1).cpu()
        for i, p in enumerate(range(start, end)):
            logprobs_matrix[p] = log_probs[i, p + 1].numpy()
        if (start // batch_size) % 10 == 0:
            elapsed = time.time() - t0
            print(f"  {end}/{seq_len}  ({elapsed:.0f}s)")
    np.save(cache_file, logprobs_matrix)
    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return logprobs_matrix

# Set RUN_FROM_SCRATCH=True to actually recompute. Default False to keep the notebook cheap.
RUN_FROM_SCRATCH = False
if RUN_FROM_SCRATCH:
    wt_seq = load_wt_sequence(f"{DATA_DIR}/sequence.fasta")
    print(f"WT length: {len(wt_seq)}")
    for name in ["esm2_t6_8M_UR50D", "esm2_t12_35M_UR50D", "esm2_t30_150M_UR50D", "esm2_t33_650M_UR50D"]:
        compute_masked_marginals(wt_seq, name, overwrite=True)

    # Recompute entropy from the freshly written log-probs
    ent_list = []
    for m in ["esm2_t6_8M_UR50D","esm2_t12_35M_UR50D","esm2_t30_150M_UR50D","esm2_t33_650M_UR50D"]:
        logp = np.load(os.path.join(CACHE_DIR, f"{m}_logprobs.npy"))
        p = np.exp(logp)
        ent_list.append(-(p * logp).sum(axis=1))
    np.save(os.path.join(CACHE_DIR, "esm2_entropy.npy"), np.mean(np.vstack(ent_list), axis=0).astype(np.float32))
    print("Re-saved esm2_entropy.npy")
else:
    print("RUN_FROM_SCRATCH=False — skipping recompute. Set to True if you want to fully redo the ESM-2 features.")

## 5. (Optional) Wider ablation — `predict_v3.py`

This is an exploratory script that tries equal-average LLR, Nelder–Mead-optimized weights, Ridge on the full LLR matrix, isotonic calibration, and Ridge/LLR blends. It uses **all available cached models** (the repo ships caches for ESM-2 8M/35M/150M/650M/3B/15B, but `predict_v3` only uses the ones whose cache files exist locally).

In [ ]:
import numpy as np, pandas as pd, os, re
from sklearn.linear_model import Ridge
from sklearn.isotonic import IsotonicRegression
from sklearn.model_selection import KFold
from scipy.stats import spearmanr
from scipy.optimize import minimize
import esm

ALL_MODELS = [
    "esm2_t6_8M_UR50D", "esm2_t12_35M_UR50D", "esm2_t30_150M_UR50D",
    "esm2_t33_650M_UR50D", "esm2_t36_3B_UR50D", "esm2_t48_15B_UR50D",
]

def parse_mutant(code):
    m = re.match(r'^([A-Z])(\d+)([A-Z])$', code)
    return m.group(1), int(m.group(2)), m.group(3)

_, alphabet = esm.pretrained.esm2_t6_8M_UR50D()
aa_to_tok = {aa: alphabet.get_idx(aa) for aa in "ACDEFGHIKLMNPQRSTVWY"}

def compute_llr_matrix(mutants):
    available, lpd = [], {}
    for m in ALL_MODELS:
        f = os.path.join("esm_cache", f"{m}_logprobs.npy")
        if os.path.exists(f):
            lpd[m] = np.load(f); available.append(m)
    M = np.zeros((len(mutants), len(available)), dtype=np.float32)
    for i, code in enumerate(mutants):
        wt_aa, pos, mut_aa = parse_mutant(code)
        for j, m in enumerate(available):
            M[i, j] = lpd[m][pos, aa_to_tok[mut_aa]] - lpd[m][pos, aa_to_tok[wt_aa]]
    return M, available

train_df = pd.read_csv("Hackathon_data/train.csv")
test_df  = pd.read_csv("Hackathon_data/test.csv")
y_train = train_df['DMS_score'].values

train_llr, available = compute_llr_matrix(train_df['mutant'].tolist())
test_llr,  _         = compute_llr_matrix(test_df['mutant'].tolist())
print(f"Models: {available}")

rho_avg = spearmanr(y_train, train_llr.mean(axis=1)).statistic
print(f"[1] Equal-average LLR (train ρ): {rho_avg:.4f}")
for j, m in enumerate(available):
    print(f"  {m}: {spearmanr(y_train, train_llr[:, j]).statistic:.4f}")

def neg_spearman(w):
    w = np.abs(w); w = w / w.sum()
    return -spearmanr(y_train, train_llr @ w).statistic

best_score, best_result = -1, None
for seed in range(50):
    rng = np.random.RandomState(seed)
    w0 = rng.dirichlet(np.ones(len(available)))
    r = minimize(neg_spearman, w0, method='Nelder-Mead', options={'maxiter': 5000, 'xatol': 1e-6})
    if -r.fun > best_score:
        best_score, best_result = -r.fun, r
w_opt = np.abs(best_result.x); w_opt /= w_opt.sum()
print(f"\n[3] Optimized weights (train ρ): {best_score:.4f}")
print("  weights:", dict(zip(available, [f"{w:.3f}" for w in w_opt])))

rhos = []
for tr, va in KFold(n_splits=5, shuffle=True, random_state=42).split(train_llr):
    rdg = Ridge(alpha=1.0).fit(train_llr[tr], y_train[tr])
    rhos.append(spearmanr(y_train[va], rdg.predict(train_llr[va])).statistic)
print(f"[4] Ridge on LLR (5-fold CV): {np.mean(rhos):.4f} ± {np.std(rhos):.4f}")

## 6. (Optional) MSA Transformer blend

Inlined version of `blend_msa.py`. Blends the base prediction (from Section 3) with the cached `esm_msa1b` per-position log-probs at several mixing weights. None of these blends beat the Section 3 best on the public leaderboard, but the analysis is included for completeness.

In [ ]:
import numpy as np, pandas as pd, re, os
from scipy.stats import spearmanr

msa_lp = np.load("esm_cache/esm_msa1b_logprobs.npy")

def msa_llr(mutants):
    out = np.zeros(len(mutants), dtype=np.float32)
    for i, c in enumerate(mutants):
        m = re.match(r'^([A-Z])(\d+)([A-Z])$', c)
        wt, pos, mut = m.group(1), int(m.group(2)), m.group(3)
        out[i] = msa_lp[pos, aa_to_tok[mut]] - msa_lp[pos, aa_to_tok[wt]]
    return out

train_msa = msa_llr(train_df['mutant'].tolist())
test_msa  = msa_llr(test_df['mutant'].tolist())
y = train_df['DMS_score'].values

print(f"Train ρ base: {spearmanr(y, train_base).statistic:.4f}")
print(f"Train ρ msa : {spearmanr(y, train_msa).statistic:.4f}")

os.makedirs("results/kaggle", exist_ok=True)
for w in [0.02, 0.05, 0.10, 0.15]:
    tr_blend = (1 - w) * train_base + w * train_msa
    te_blend = (1 - w) * test_base  + w * test_msa
    rho = spearmanr(y, tr_blend).statistic
    tag = f"msa_blend_w{int(w*100):02d}"
    pd.DataFrame({"id": np.arange(len(test_df)), "DMS_score": te_blend}).to_csv(f"results/kaggle/kaggle_{tag}.csv", index=False)
    pd.DataFrame({"mutant": test_df['mutant'], "DMS_score_predicted": te_blend}).to_csv(f"predictions_{tag}.csv", index=False)
    print(f"w={w:.2f}  train ρ={rho:.4f}")

## 7. Final outputs

After running Sections 1–3 you will have these submission files in the working directory:

- `predictions_q1cal_w10.csv`
- `predictions_q1cal_w20.csv`  ← **best public-leaderboard submission (ρ ≈ 0.41266)**
- `predictions_q1cal_w30.csv`

And the corresponding Kaggle-format CSVs under `results/kaggle/`.

In [ ]:
!ls -la predictions_*.csv results/kaggle/ 2>/dev/null